# Pitch Vision — Phase 0 & 1 (v3 — bigger imgsz so the ball is actually visible to the model)

**What changed from v2:** v2 fixed player detection (drone-matched training data) but the ball was still missed almost every frame — because it starts at only ~5px in our stored 1920x1080 frames, and training's default `imgsz=640` shrinks that down to roughly **1.6 pixels**, which is too small for the model to learn from. v3 trains at `imgsz=1280` instead, so the ball keeps close to 3.3px of real signal — a meaningful jump, cheap to test before reaching for anything heavier (like tiling the frames).

To keep this from crashing Colab's free-tier RAM the way earlier attempts did, batch size and worker count are both turned down this time (`batch=4`, `workers=1`) to offset the extra memory `imgsz=1280` uses per image. Slower, not riskier — that trade is fine since speed isn't the concern here.

**Before running:** Runtime → Change runtime type → GPU (T4, free tier).

You'll need: `soccertrack_yolo_dataset.zip` from your project's `training/` folder, uploaded to your Google Drive (the plain Colab upload widget timed out on this file last time — Drive handles the size fine).

In [ ]:
!pip install -q ultralytics supervision
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

## Part A — Baseline (optional, skip if you already have it)

Skip this if you already have `baseline_output.zip` from the v1 run. Only run it if you want to redo the stock-YOLO comparison fresh.

In [ ]:
from google.colab import files
print("Upload clip_01_0-12s.mp4 (from your project's input_videos/ folder):")
uploaded = files.upload()
clip_path = list(uploaded.keys())[0]
print("Using:", clip_path)

## Part B — Fine-tune YOLO, this time with enough resolution for the ball

Same drone-matched dataset as v2 (7 clips, every 12th frame, 2 classes: `player`, `ball`) — only the training resolution and batch settings changed.

In [ ]:
# Upload soccertrack_yolo_dataset.zip to your Google Drive first (drag-and-drop at drive.google.com —
# handles large files reliably, unlike Colab's own upload widget). This mounts your Drive and copies
# it in from there instead.
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/soccertrack_yolo_dataset.zip" /content/
uploaded_ds = {'soccertrack_yolo_dataset.zip': None}  # keeps the next cell's logic unchanged
print("Copied from Drive.")

In [ ]:
import zipfile, os

zip_name = [k for k in uploaded_ds.keys() if k.endswith('.zip')][0]
extract_dir = '/content/soccertrack_yolo_dataset'
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

train_dir = os.path.join(extract_dir, 'images', 'train')
val_dir = os.path.join(extract_dir, 'images', 'valid')
print("train exists:", os.path.isdir(train_dir), "-", len(os.listdir(train_dir)) if os.path.isdir(train_dir) else 0, "images")
print("valid exists:", os.path.isdir(val_dir), "-", len(os.listdir(val_dir)) if os.path.isdir(val_dir) else 0, "images")

# Rewrite data.yaml with the actual Colab-side paths (this is the same bug class that bit us with
# the Roboflow download originally — always regenerate paths after extracting, never trust the
# paths baked into a downloaded/extracted data.yaml).
data_yaml = os.path.join(extract_dir, 'data.yaml')
with open(data_yaml, 'w') as f:
    f.write(f"train: {train_dir}\nval: {val_dir}\nnc: 2\nnames: ['player', 'ball']\n")
print("data.yaml at:", data_yaml)
!cat {data_yaml}

In [ ]:
# imgsz bumped 640 -> 1280 so the ~5px ball in our source frames doesn't shrink below ~2px.
# batch dropped 8 -> 4 and workers dropped 2 -> 1 to offset the extra memory imgsz=1280 uses per
# image — this is the RAM-safety trade for the resolution increase. Slower, but shouldn't crash.
# If Colab disconnects mid-run, re-run this cell with resume=True to continue from the last checkpoint.
!yolo task=detect mode=train model=yolov8m.pt data="{data_yaml}" epochs=100 imgsz=1280 batch=4 workers=1

In [ ]:
# Grab the trained weights — this is the one file you need to bring back to your local project's models/ folder.
import glob, shutil
best_pt = glob.glob('runs/detect/train*/weights/best.pt')[-1]
shutil.copy(best_pt, 'best.pt')
files.download('best.pt')
print("Downloaded best.pt — save this into your local project's models/ folder as best_topview.pt.")

## Validate: is the ball actually showing up now?

Re-run on the exact same clip. Compare the `ball` avg/frame number against v2's (5 detections total over 362 frames) — that's the number that should move.

In [ ]:
from ultralytics import YOLO
import glob, shutil
from google.colab import files

finetuned_model = YOLO('best.pt')

class_counts = {}
total_ft_frames = 0

# stream=True processes and discards each frame's result as it goes, instead of holding
# the whole video's detections in RAM at once.
for r in finetuned_model.predict(source=clip_path, save=True, conf=0.25, stream=True):
    total_ft_frames += 1
    for c in r.boxes.cls:
        name = finetuned_model.names[int(c)]
        class_counts[name] = class_counts.get(name, 0) + 1

print("Classes this model knows:", finetuned_model.names)
print(f"\nTotal frames: {total_ft_frames}")
for name, count in class_counts.items():
    print(f"  {name}: {count} detections total, {count/total_ft_frames:.1f} avg/frame")

out_dir = glob.glob('runs/detect/predict*')[-1]
shutil.make_archive('finetuned_output_v3', 'zip', out_dir)
files.download('finetuned_output_v3.zip')

### Done with Colab for now

You should have downloaded: `best.pt`, `finetuned_output_v3.zip` (and `baseline_output.zip` if you ran Part A).

Save `best.pt` as `models/best_topview.pt` locally (replacing the v2 attempt) — that's what Stage 2 (tracking) will load.

**If the ball avg/frame is still very low:** that's the signal to move to tiling (splitting frames into smaller crops so the ball never shrinks in the first place) rather than pushing `imgsz` even higher — at some point a bigger `imgsz` stops helping and just costs more RAM/time. Tell me the new number either way and we'll decide together.